# A02 — Antireflection film

## Mathematical and physical model

Use time dependence $e^{-i\omega t}$, vacuum wavenumber $k_0=2\pi/\lambda_0$ and conserved transverse wavevector $k_\parallel$. For each isotropic, nonmagnetic layer set
$$
u_j=k_{z,j}/k_0=\sqrt{\epsilon_j-(k_\parallel/k_0)^2},\qquad
Y_{s,j}=u_j,\qquad Y_{p,j}=\epsilon_j/u_j .
$$
Choose the forward branch with $\operatorname{Im}u_j\ge 0$ for passive layers. The admittances apply to **tangential** electric-field coefficients. The notebook compares the solver outputs only after converting them to the common ray-based polarization and power conventions.

Let layer $1$ be the coating, with thickness $d$ and phase $\delta=k_0u_1d$. With tangential Fresnel coefficients $r_{ij}=(Y_i-Y_j)/(Y_i+Y_j)$, the reflected amplitude of the coated interface is
$$
r=\frac{r_{01}+r_{12}e^{2i\delta}}{1+r_{01}r_{12}e^{2i\delta}}.
$$
At normal incidence, $n_1^2=n_0n_2$ and $n_1d=\lambda_{\mathrm{design}}/4$ give cancellation at 550 nm.

## Physical scenario

A single lossless dielectric film separates air and glass. Its optical thickness is one quarter of the 550 nm design wavelength. At normal incidence the two main reflected fields cancel near the design point, while the transmitted field accumulates a wavelength-dependent phase.

## Possible applications

Models the basic mechanism of single-layer antireflection coatings on lenses, windows, and sensor covers. This is an idealized index-matching example for checking spectral reflectance and transmitted phase.

## Benchmark objective

Check the reflection minimum and complex transmission against an independent thin-film calculation.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#a02)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("A02")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## Analytic multilayer reference

In [ ]:
from rcwa_cv.references.multilayer import solve as solve_reference

results["reference"] = solve_sweep(solve_reference, samples, settings)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare with Lumerical

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a shared excitation point. The table and difference plots compare each open-source solver with Lumerical.
Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)